# Lab 47: The Rubber Hand

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 6, *The Embodied Avatar*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-47.ipynb)

**What you will do:** change the width and centre of the brain's temporal binding window to see how much delay between touch and vision the model will still fuse into one event.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 47 you placed your real hand out of sight and had a friend brush a visible rubber hand
and your hidden hand at exactly the same time and place. After about a minute of this
synchronous stroking, many people start to feel that the touch is coming from the rubber
hand -- as if it were their own. You then repeated the stroking asynchronously (at different
times), and for most people the feeling of ownership collapsed.

## Record your results

Researchers often measure the rubber hand illusion with a short questionnaire, rating
agreement with a few statements straight after each stroking condition. Rate each statement
from 1 (strongly disagree) to 7 (strongly agree) -- once for the synchronous stroking in
Step 4, and once for the asynchronous stroking in Step 5. The first three statements describe
the illusion itself; the last three are control statements that should stay low regardless of
synchrony, so you can check you are not just agreeing with everything.

In [ ]:
ratings = pd.DataFrame({
    "statement": [
        "I felt the touch where I saw the rubber hand touched",
        "It felt like the rubber hand was my own hand",
        "I would have flinched if the rubber hand were struck",
        "My real hand felt like it was turning to rubber",     # control
        "It seemed like I had two right hands",                # control
        "The rubber hand started to look like my real skin",   # control
    ],
    "is_ownership_item": [True, True, True, False, False, False],
    "synchronous (1-7)": [6, 6, 5, 2, 1, 2],    # example data: replace with your own ratings
    "asynchronous (1-7)": [2, 2, 1, 2, 1, 2],   # example data: replace with your own ratings
})
display(ratings)

def ownership_score(col):
    own = ratings.loc[ratings.is_ownership_item, col].mean()
    ctrl = ratings.loc[~ratings.is_ownership_item, col].mean()
    return own - ctrl

scores = pd.Series({"Synchronous stroking": ownership_score("synchronous (1-7)"),
                     "Asynchronous stroking": ownership_score("asynchronous (1-7)")})
display(scores.to_frame("ownership score (illusion items - control items)"))
scores.plot.barh(color=["steelblue", "grey"], figsize=(6, 2))
plt.xlabel("ownership score"); plt.axvline(0, color="black", linewidth=0.8); plt.show()
print("A clearly higher score for synchronous than asynchronous stroking matches the book's claim that")
print("synchrony -- not just watching a rubber hand being touched -- drives the illusion.")

## Simulation

The rubber hand illusion works because the brain integrates visual and tactile signals that arrive close together in time. But how close is "close enough"? This simulation models the brain's temporal binding window—the time range within which signals from different senses are fused into a single percept.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

delays = np.linspace(-500, 500, 1000)  # ms between touch and vision

# The brain's binding window: a Gaussian centred near 0 ms
# (slightly shifted toward vision-first, reflecting visual dominance)
binding_prob = np.exp(-0.5 * ((delays - 30) / 120) ** 2)

# Simulate trials: does the brain bind the two signals?
n_trials = 200
test_delays = np.array([-300, -150, -50, 0, 50, 150, 300])
results = {}
for d in test_delays:
    prob = np.exp(-0.5 * ((d - 30) / 120) ** 2)
    bound = np.random.binomial(1, prob, n_trials)
    results[d] = np.mean(bound)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(delays, binding_prob, "steelblue", linewidth=2)
ax1.fill_between(delays, binding_prob, alpha=0.2, color="steelblue")
ax1.axvline(0, color="grey", linestyle=":", alpha=0.5)
ax1.set_xlabel("Delay between touch and vision (ms)")
ax1.set_ylabel("Probability of integration")
ax1.set_title("The Temporal Binding Window")

ax2.bar([str(d) for d in test_delays],
    [results[d] for d in test_delays], color="steelblue", alpha=0.7)
ax2.set_xlabel("Delay (ms)")
ax2.set_ylabel("Proportion 'bound'")
ax2.set_title(f"Simulated Binding ({n_trials} trials per delay)")
ax2.axhline(0.5, color="orange", linestyle="--", label="50% threshold")
ax2.legend()

plt.tight_layout()
plt.show()

print("Within ±150 ms: brain fuses touch and vision → rubber hand illusion possible")
print("Beyond ±300 ms: signals feel separate → illusion breaks down")

**What to notice:** The binding window is roughly ±150 ms wide, and slightly asymmetric (vision can lag behind touch more than the reverse). Outside this window, the two signals are perceived as separate events. This is why perfectly synchronous brushstrokes on the rubber hand are essential—even a small delay breaks the illusion.

## Change the parameters

**centre** is the delay (in ms) at which binding is most likely -- the lab's default of
30 ms reflects a slight bias toward vision arriving first. **width** controls how forgiving
the window is. **n_trials** controls how many simulated trials are run at each test delay.
(If the sliders do not appear, run the cell again.)

In [ ]:
from ipywidgets import interact, IntSlider

def explore(centre=30, width=120, n_trials=200):
    np.random.seed(42)
    delays = np.linspace(-500, 500, 1000)
    binding_prob = np.exp(-0.5 * ((delays - centre) / width) ** 2)

    test_delays = np.array([-300, -150, -50, 0, 50, 150, 300])
    results = {}
    for d in test_delays:
        prob = np.exp(-0.5 * ((d - centre) / width) ** 2)
        bound = np.random.binomial(1, prob, n_trials)
        results[d] = np.mean(bound)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(delays, binding_prob, "steelblue", linewidth=2)
    ax1.fill_between(delays, binding_prob, alpha=0.2, color="steelblue")
    ax1.axvline(0, color="grey", linestyle=":", alpha=0.5)
    ax1.set_xlabel("Delay between touch and vision (ms)")
    ax1.set_ylabel("Probability of integration")
    ax1.set_title(f"Binding window: centre={centre} ms, width={width} ms")

    ax2.bar([str(d) for d in test_delays], [results[d] for d in test_delays], color="steelblue", alpha=0.7)
    ax2.set_xlabel("Delay (ms)")
    ax2.set_ylabel("Proportion 'bound'")
    ax2.set_title(f"Simulated binding ({n_trials} trials per delay)")
    ax2.axhline(0.5, color="orange", linestyle="--", label="50% threshold")
    ax2.legend()
    plt.tight_layout()
    plt.show()

interact(explore,
         centre=IntSlider(value=30, min=-100, max=100, step=10),
         width=IntSlider(value=120, min=30, max=300, step=10),
         n_trials=IntSlider(value=200, min=50, max=500, step=50));

## Questions to think about

1. Shrink width down to about 30 ms. At roughly what delay does the binding probability fall below 50 percent? How does that compare with the "roughly plus-or-minus 150 ms" figure quoted in the lab?
2. Move centre away from 0, say to 100. What does a non-zero centre mean physically -- does the model tolerate touch-before-vision delays or vision-before-touch delays better at that setting? Does it match the lab's note about a "slight" visual-dominance bias, or does it overstate it?
3. With width widened to 300 ms, would asynchronous stroking (Step 5 of the rubber-hand instructions) still be expected to break the illusion? Why might a real brain need a narrower window than this?
4. Increase n_trials substantially. Does the smooth curve on the left change shape, and does the bar chart on the right get smoother or stay just as noisy? What is the difference between the underlying fixed probability and a simulated sample of trials?

## Challenge

The lab's binding-window logic applies to any pair of senses, not just touch and vision.
Estimate the audio-visual lag you have noticed on a laggy video call (or measure one with a
stopwatch on a recorded clip), and use the code below -- which reuses the same Gaussian
binding-window formula -- to estimate whether the brain would still fuse the two streams.

In [ ]:
# Example data: replace with your own estimate or measurement of an audio-visual lag.
video_call_lag_ms = 180   # example: the lag you noticed on a laggy call, in milliseconds

centre, width = 30, 120
prob = np.exp(-0.5 * ((video_call_lag_ms - centre) / width) ** 2)
print(f"Estimated probability of fusing audio and video at a {video_call_lag_ms} ms lag: {prob:.0%}")
print("This reuses the same Gaussian binding-window model as the rubber hand simulation above,")
print("applied to a different pair of senses. The lag value is example data -- use your own.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-47.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")